In [2]:
import tiktoken

dir(tiktoken)

['Encoding',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 '__version__',
 '_tiktoken',
 'core',
 'encoding_for_model',
 'encoding_name_for_model',
 'get_encoding',
 'list_encoding_names',
 'model',
 'registry']

In [3]:
help(tiktoken.encoding_for_model)

Help on function encoding_for_model in module tiktoken.model:

encoding_for_model(model_name: 'str') -> 'Encoding'
    Returns the encoding used by a model.

    Raises a KeyError if the model name is not recognised.



In [5]:
encoding = tiktoken.encoding_for_model("gpt-4.1-mini")

help(encoding)

Help on Encoding in module tiktoken.core object:

class Encoding(builtins.object)
 |  Encoding(name: 'str', *, pat_str: 'str', mergeable_ranks: 'dict[bytes, int]', special_tokens: 'dict[str, int]', explicit_n_vocab: 'int | None' = None)
 |
 |  Methods defined here:
 |
 |  __getstate__(self) -> 'object'
 |      Helper for pickle.
 |
 |  __init__(self, name: 'str', *, pat_str: 'str', mergeable_ranks: 'dict[bytes, int]', special_tokens: 'dict[str, int]', explicit_n_vocab: 'int | None' = None)
 |      Creates an Encoding object.
 |
 |      See openai_public.py for examples of how to construct an Encoding object.
 |
 |      Args:
 |          name: The name of the encoding. It should be clear from the name of the encoding
 |              what behaviour to expect, in particular, encodings with different special tokens
 |              should have different names.
 |          pat_str: A regex pattern string that is used to split the input text.
 |          mergeable_ranks: A dictionary mapping 

In [6]:
tokens = encoding.encode("Hi my name is oink")

tokens

[12194, 922, 1308, 382, 293, 881]

In [7]:
for token_id in tokens:
  token_text = encoding.decode([token_id])
  print(f"{token_id} = {token_text}")

12194 = Hi
922 =  my
1308 =  name
382 =  is
293 =  o
881 = ink


In [8]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
model_name = os.getenv('MODEL_NAME')
base_url = os.getenv('OLLAMA_BASE_URL')

if not model_name and not base_url:
  print("No MODEL_NAME and OLLAMA_BASE_URL found!")
else:
  print("MODEL_NAME and OLLAMA_BASE_URL found it looks good so far!")

MODEL_NAME and OLLAMA_BASE_URL found it looks good so far!


In [18]:
from openai import OpenAI

openai = OpenAI(
  base_url=base_url,
  api_key=model_name
)

In [19]:
user_message = "hi my name is oink"

messages = [
  {"role": "system", "content" : "you are an helpful assistance"},
  {"role": "user", "content" : user_message}
  ]

In [23]:
help(openai.chat.completions.create)

Help on method create in module openai.resources.chat.completions.completions:

create(*, messages: 'Iterable[ChatCompletionMessageParam]', model: 'Union[str, ChatModel]', audio: 'Optional[ChatCompletionAudioParam] | Omit' = <openai.Omit object at 0x00000164D7636510>, frequency_penalty: 'Optional[float] | Omit' = <openai.Omit object at 0x00000164D7636510>, function_call: 'completion_create_params.FunctionCall | Omit' = <openai.Omit object at 0x00000164D7636510>, functions: 'Iterable[completion_create_params.Function] | Omit' = <openai.Omit object at 0x00000164D7636510>, logit_bias: 'Optional[Dict[str, int]] | Omit' = <openai.Omit object at 0x00000164D7636510>, logprobs: 'Optional[bool] | Omit' = <openai.Omit object at 0x00000164D7636510>, max_completion_tokens: 'Optional[int] | Omit' = <openai.Omit object at 0x00000164D7636510>, max_tokens: 'Optional[int] | Omit' = <openai.Omit object at 0x00000164D7636510>, metadata: 'Optional[Metadata] | Omit' = <openai.Omit object at 0x00000164D7636

In [24]:
response = openai.chat.completions.create(model=model_name, messages=messages)

In [31]:
help(response.choices[0].message)

Help on ChatCompletionMessage in module openai.types.chat.chat_completion_message object:

class ChatCompletionMessage(openai.BaseModel)
 |  ChatCompletionMessage(*, content: Optional[str] = None, refusal: Optional[str] = None, role: Literal['assistant'], annotations: Optional[List[openai.types.chat.chat_completion_message.Annotation]] = None, audio: Optional[openai.types.chat.chat_completion_audio.ChatCompletionAudio] = None, function_call: Optional[openai.types.chat.chat_completion_message.FunctionCall] = None, tool_calls: Optional[List[Annotated[Union[openai.types.chat.chat_completion_message_function_tool_call.ChatCompletionMessageFunctionToolCall, openai.types.chat.chat_completion_message_custom_tool_call.ChatCompletionMessageCustomToolCall], PropertyInfo(alias='None', format=None, format_template='None', discriminator='type')]]] = None, **extra_data: Any) -> None
 |
 |  A chat completion message generated by the model.
 |
 |  Method resolution order:
 |      ChatCompletionMessage

In [30]:
response.choices[0].message.content

'Hi Oink! How can I assist you today?'

In [ ]:
#illution of memory the follow up doesnt add up to the previous output so llm dont have memory every call in llm is Stateless we have to give impression that llm has a memory

messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What's my name?"}
  ]

In [33]:
response = openai.chat.completions.create(model=model_name,messages=messages)

response.choices[0].message.content

"I'm sorry, but I don't have access to that information. How can I assist you today?"

In [34]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": user_message},
    {"role": "assistant", "content": "Hi Ed! How can I assist you today?"},
    {"role": "user", "content": "What's my name?"}
  ]

In [35]:
response = openai.chat.completions.create(model=model_name, messages= messages)

response.choices[0].message.content

'Your name is Oink.'

## To recap

1. Every call to an LLM is stateless
2. We pass in the entire conversation so far in the input prompt, every time
3. This gives the illusion that the LLM has memory - it apparently keeps the context of the conversation
4. But this is a trick; it's a by-product of providing the entire conversation, every time
5. An LLM just predicts the most likely next tokens in the sequence; if that sequence contains "My name is Ed" and later "What's my name?" then it will predict.. Ed!

The ChatGPT product uses exactly this trick - every time you send a message, it's the entire conversation that gets passed in.

"Does that mean we have to pay extra each time for all the conversation so far"

For sure it does. And that's what we WANT. We want the LLM to predict the next tokens in the sequence, looking back on the entire conversation. We want that compute to happen, so we need to pay the electricity bill for it!
